# TabPFN → drzewo decyzyjne (rozwiązanie 2)

Colab: **Runtime → Change runtime type → T4 GPU**, potem Run all.

TabPFN uczy się z `val.csv`, etykietuje `train.csv`, a sklearn-drzewo destyluje te decyzje na nazwanych sygnaturach akustycznych. Aplikacja warsztatowa (`app_tabpfn.py`) liczy już tylko drzewo — CPU, ścieżka if/then, punkty za explainability.

**Weryfikacja:** trening wyłącznie na `val.csv`. Uczciwy Raw_Score liczymy na `final_valid.csv` (silniki, których model nie widział). `test.csv` zostaje submitem bez etykiet.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q tabpfn scikit-learn pandas numpy torch joblib plotly
    from google.colab import files
    print("Wgraj val.csv, final_valid.csv, train.csv, test.csv oraz tabpfn_diagnose.py")
    files.upload()

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.1/762.1 kB 20.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.5 MB/s eta 0:00:00
Wgraj val.csv, train.csv, test.csv oraz tabpfn_diagnose.py


In [ ]:
from tabpfn_diagnose import TabPFNTreeDiagnoser, pick_device, print_eval, read_labeled_csv
import pandas as pd

val = read_labeled_csv("val.csv")
holdout = read_labeled_csv("final_valid.csv")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
overlap = set(val["engine_id"]) & set(holdout["engine_id"])
assert not overlap, f"wyciek silników val ∩ final_valid: {sorted(overlap)}"
print(
    "val", val.shape, "final_valid", holdout.shape,
    "train", train.shape, "test", test.shape,
    "device=", pick_device(),
)
print("silniki holdout:", sorted(holdout["engine_id"].unique()))

## Nauczyciel + destylacja

Fit tylko na `val.csv` (+ pseudo-etykiety z `train.csv`). Na T4 włącz CV w komórce poniżej (GroupKFold po silniku na val). Na CPU zostaw `do_cv=False`. Score na `final_valid` jest w następnej sekcji.

In [ ]:
device = pick_device()
n_estimators = 8 if device == "cuda" else 4
do_cv = device == "cuda"  # T4: extra GroupKFold na val; CPU: pomiń

model = TabPFNTreeDiagnoser().fit(
    val,
    train,
    device=device,
    n_estimators=n_estimators,
    do_cv=do_cv,
)
model.save()
print(model.meta)

## Holdout: `final_valid.csv`

Silniki spoza `val.csv`. To jest uczciwy Raw_Score (to samo, co liczy jury, tylko na naszym holdoucie).

In [ ]:
sub_ho_tree = model.predict(holdout)
sub_ho_tabpfn = model.predict_teacher(holdout, model.teacher_)
y_ho = holdout["label"].to_numpy()
s_ho = holdout["severity"].to_numpy()
print_eval("final_valid — TabPFN teacher", y_ho, sub_ho_tabpfn["label"].to_numpy(), s_ho, sub_ho_tabpfn["severity"].to_numpy())
print_eval("final_valid — distilled tree", y_ho, sub_ho_tree["label"].to_numpy(), s_ho, sub_ho_tree["severity"].to_numpy())
agree_ho = (sub_ho_tree["label"] == sub_ho_tabpfn["label"]).mean()
print(f"zgoda drzewo vs TabPFN na final_valid: {agree_ho:.3f}")

## Drzewo, które zobaczy mechanik

In [ ]:
print(model.rules_text())

## Submit `test.csv` + zgodność nauczyciel / student

Ten sam model co na `final_valid` (fit na `val.csv`). `test.csv` nie ma etykiet.

In [ ]:
sub_tree = model.predict(test)
sub_tabpfn = model.predict_teacher(test, model.teacher_)
sub_tree.to_csv("predictions_tree.csv", index=False)
sub_tabpfn.to_csv("predictions_tabpfn.csv", index=False)
agree = (sub_tree["label"] == sub_tabpfn["label"]).mean()
print(f"zgoda drzewo vs TabPFN na teście (bez etykiet): {agree:.3f}")
print("TabPFN\n", sub_tabpfn["label"].value_counts())
print("drzewo\n", sub_tree["label"].value_counts())

if IN_COLAB:
    files.download("predictions_tabpfn.csv")
    files.download("predictions_tree.csv")
    files.download("artifacts/diagnoser_tree.joblib")

Lokalnie po pobraniu `diagnoser_tree.joblib` do `artifacts/`:

```bash
streamlit run app_tabpfn.py
```